**Step 1**

In [1]:
import requests
import zipfile
import os

url = "https://cdn.freecodecamp.org/project-data/books/book-crossings.zip"

response = requests.get(url)

with open("book-crossings.zip", "wb") as f:
    f.write(response.content)

with zipfile.ZipFile("book-crossings.zip", "r") as zip_ref:
    zip_ref.extractall("dataset")

print("Dataset downloaded successfully!")

Dataset downloaded successfully!


In [2]:
os.listdir("dataset")

['BX-Users.csv', 'BX-Book-Ratings.csv', 'BX-Books.csv']

In [4]:
os.makedirs("bronze", exist_ok=True)
os.makedirs("silver", exist_ok=True)
os.makedirs("gold", exist_ok=True)


In [5]:
import pandas as pd
from datetime import datetime
users = pd.read_csv(
    "dataset/BX-Users.csv",
    sep=";",
    encoding="ISO-8859-1"
)

books = pd.read_csv(
    "dataset/BX-Books.csv",
    sep=";",
    encoding="ISO-8859-1",
    on_bad_lines="warn"
)

ratings = pd.read_csv(
    "dataset/BX-Book-Ratings.csv",
    sep=";",
    encoding="ISO-8859-1"
)

print("Users:", users.shape)
print("Books:", books.shape)
print("Ratings:", ratings.shape)

/tmp/ipykernel_2784/4286641348.py:9: ParserWarning: Skipping line 6452: expected 8 fields, saw 9
Skipping line 43667: expected 8 fields, saw 10
Skipping line 51751: expected 8 fields, saw 9

  books = pd.read_csv(
/tmp/ipykernel_2784/4286641348.py:9: ParserWarning: Skipping line 92038: expected 8 fields, saw 9
Skipping line 104319: expected 8 fields, saw 9
Skipping line 121768: expected 8 fields, saw 9

  books = pd.read_csv(
/tmp/ipykernel_2784/4286641348.py:9: ParserWarning: Skipping line 144058: expected 8 fields, saw 9
Skipping line 150789: expected 8 fields, saw 9
Skipping line 157128: expected 8 fields, saw 9
Skipping line 180189: expected 8 fields, saw 9
Skipping line 185738: expected 8 fields, saw 9

  books = pd.read_csv(
/tmp/ipykernel_2784/4286641348.py:9: ParserWarning: Skipping line 209388: expected 8 fields, saw 9
Skipping line 220626: expected 8 fields, saw 9
Skipping line 227933: expected 8 fields, saw 11
Skipping line 228957: expected 8 fields, saw 10
Skipping line 245

Users: (278858, 3)
Books: (271360, 8)
Ratings: (1149780, 3)


In [6]:

import numpy as np


users = pd.read_csv(
    "dataset/BX-Users.csv",
    sep=";",
    encoding="ISO-8859-1"
)

books = pd.read_csv(
    "dataset/BX-Books.csv",
    sep=";",
    encoding="ISO-8859-1",
    on_bad_lines="warn",
    low_memory=False
)

ratings = pd.read_csv(
    "dataset/BX-Book-Ratings.csv",
    sep=";",
    encoding="ISO-8859-1"
)

print("Users:", users.shape)
print("Books:", books.shape)
print("Ratings:", ratings.shape)

/tmp/ipykernel_2784/2443307738.py:10: ParserWarning: Skipping line 6452: expected 8 fields, saw 9
Skipping line 43667: expected 8 fields, saw 10
Skipping line 51751: expected 8 fields, saw 9
Skipping line 92038: expected 8 fields, saw 9
Skipping line 104319: expected 8 fields, saw 9
Skipping line 121768: expected 8 fields, saw 9
Skipping line 144058: expected 8 fields, saw 9
Skipping line 150789: expected 8 fields, saw 9
Skipping line 157128: expected 8 fields, saw 9
Skipping line 180189: expected 8 fields, saw 9
Skipping line 185738: expected 8 fields, saw 9
Skipping line 209388: expected 8 fields, saw 9
Skipping line 220626: expected 8 fields, saw 9
Skipping line 227933: expected 8 fields, saw 11
Skipping line 228957: expected 8 fields, saw 10
Skipping line 245933: expected 8 fields, saw 9
Skipping line 251296: expected 8 fields, saw 9
Skipping line 259941: expected 8 fields, saw 9
Skipping line 261529: expected 8 fields, saw 9

  books = pd.read_csv(


Users: (278858, 3)
Books: (271360, 8)
Ratings: (1149780, 3)


In [8]:
timestamp = datetime.now()

users["ingestion_timestamp"] = timestamp
books["ingestion_timestamp"] = timestamp
ratings["ingestion_timestamp"] = timestamp


In [9]:
!pip install pyarrow

In [13]:
users.to_parquet("bronze/users_bronze.parquet", index=False)

books.to_parquet("bronze/books_bronze.parquet", index=False)

ratings.to_parquet("bronze/ratings_bronze.parquet", index=False)

print(os.listdir("bronze"))

['users_bronze.parquet', 'ratings_bronze.parquet', 'books_bronze.parquet']


In [14]:
users_bronze = pd.read_parquet("bronze/users_bronze.parquet")
books_bronze = pd.read_parquet("bronze/books_bronze.parquet")
ratings_bronze = pd.read_parquet("bronze/ratings_bronze.parquet")

print("Users:", users_bronze.shape)
print("Books:", books_bronze.shape)
print("Ratings:", ratings_bronze.shape)

Users: (278858, 4)
Books: (271360, 9)
Ratings: (1149780, 4)


**Step 2**

In [15]:
users = pd.read_parquet("bronze/users_bronze.parquet")
books = pd.read_parquet("bronze/books_bronze.parquet")
ratings = pd.read_parquet("bronze/ratings_bronze.parquet")

In [16]:
tables = {
    "Users": users,
    "Books": books,
    "Ratings": ratings
}

for name, df in tables.items():
    print("\n==========", name, "==========")

    print("\nShape:")
    print(df.shape)

    print("\nData Types:")
    print(df.dtypes)

    print("\nMissing Values:")
    print(df.isnull().sum())


========== Users ==========

Shape:
(278858, 4)

Data Types:
User-ID                         int64
Location                       object
Age                           float64
ingestion_timestamp    datetime64[us]
dtype: object

Missing Values:
User-ID                     0
Location                    0
Age                    110762
ingestion_timestamp         0
dtype: int64

========== Books ==========

Shape:
(271360, 9)

Data Types:
ISBN                           object
Book-Title                     object
Book-Author                    object
Year-Of-Publication            object
Publisher                      object
Image-URL-S                    object
Image-URL-M                    object
Image-URL-L                    object
ingestion_timestamp    datetime64[us]
dtype: object

Missing Values:
ISBN                   0
Book-Title             0
Book-Author            2
Year-Of-Publication    0
Publisher              2
Image-URL-S            0
Image-URL-M            0
Image-URL-L 

In [18]:
rating_counts = ratings["Book-Rating"].value_counts().sort_index()

print(rating_counts)

print("Number of zero ratings:")
print((ratings["Book-Rating"] == 0).sum())

Book-Rating
0     716109
1       1770
2       2759
3       5996
4       8904
5      50974
6      36924
7      76457
8     103736
9      67541
10     78610
Name: count, dtype: int64
Number of zero ratings:
716109


In [21]:
print("Age Statistics:")
print(users["Age"].describe())

print("Missing Ages:")
print(users["Age"].isnull().sum())

print("Ages below 5:")
print((users["Age"] < 5).sum())

print("Ages above 100:")
print((users["Age"] > 100).sum())

Age Statistics:
count    168096.000000
mean         34.751434
std          14.428097
min           0.000000
25%          24.000000
50%          32.000000
75%          44.000000
max         244.000000
Name: Age, dtype: float64
Missing Ages:
110762
Ages below 5:
882
Ages above 100:
366


In [22]:
books["Year-Of-Publication"] = pd.to_numeric(
    books["Year-Of-Publication"],
    errors="coerce"
)

current_year = datetime.now().year

print("Publication Year Statistics:")
print(books["Year-Of-Publication"].describe())

print("Publication year 0:")
print((books["Year-Of-Publication"] == 0).sum())

print("Future publication years:")
print((books["Year-Of-Publication"] > current_year).sum())

print("Missing publication years:")
print(books["Year-Of-Publication"].isnull().sum())

Publication Year Statistics:
count    271357.000000
mean       1959.760817
std         257.994226
min           0.000000
25%        1989.000000
50%        1995.000000
75%        2000.000000
max        2050.000000
Name: Year-Of-Publication, dtype: float64
Publication year 0:
4618
Future publication years:
11
Missing publication years:
3


In [23]:
ratings_per_user = ratings.groupby("User-ID").size()

print(ratings_per_user.describe())

count    105283.000000
mean         10.920851
std          90.562825
min           1.000000
25%           1.000000
50%           1.000000
75%           4.000000
max       13602.000000
dtype: float64


In [24]:
ratings_per_book = ratings.groupby("ISBN").size()

print(ratings_per_book.describe())

count    340556.000000
mean          3.376185
std          12.436252
min           1.000000
25%           1.000000
50%           1.000000
75%           2.000000
max        2502.000000
dtype: float64


In [25]:
total_users = ratings["User-ID"].nunique()
total_books = ratings["ISBN"].nunique()

total_possible_ratings = total_users * total_books

actual_ratings = ratings.drop_duplicates(
    subset=["User-ID", "ISBN"]
).shape[0]

sparsity = (
    1 - (actual_ratings / total_possible_ratings)
) * 100

print("Total Users:", total_users)
print("Total Books:", total_books)
print("Actual User-Book Ratings:", actual_ratings)
print("Matrix Sparsity:", round(sparsity, 2), "%")

Total Users: 105283
Total Books: 340556
Actual User-Book Ratings: 1149780
Matrix Sparsity: 100.0 %


**Step 3**

In [26]:

users = pd.read_parquet("bronze/users_bronze.parquet")
books = pd.read_parquet("bronze/books_bronze.parquet")
ratings = pd.read_parquet("bronze/ratings_bronze.parquet")

In [28]:
users = users.drop_duplicates(subset=["User-ID"])
books = books.drop_duplicates(subset=["ISBN"])
ratings = ratings.drop_duplicates()

In [29]:
users["Age"] = pd.to_numeric(users["Age"], errors="coerce")

users.loc[
    (users["Age"] < 5) | (users["Age"] > 100),
    "Age"
] = np.nan

median_age = users["Age"].median()

users["Age"] = users["Age"].fillna(median_age)

In [30]:
users["Location"] = users["Location"].fillna("unknown").astype(str)

users["Location"] = users["Location"].str.strip().str.lower()

location_parts = users["Location"].str.split(",", n=2, expand=True)

users["City"] = location_parts[0].str.strip()

users["State"] = location_parts[1].str.strip() if 1 in location_parts.columns else "unknown"

users["Country"] = location_parts[2].str.strip() if 2 in location_parts.columns else "unknown"

users["City"] = users["City"].replace("", "unknown").fillna("unknown")
users["State"] = users["State"].replace("", "unknown").fillna("unknown")
users["Country"] = users["Country"].replace("", "unknown").fillna("unknown")

users = users.drop(columns=["Location"])

In [31]:
books["Year-Of-Publication"] = pd.to_numeric(
    books["Year-Of-Publication"],
    errors="coerce"
)

current_year = datetime.now().year

books.loc[
    (books["Year-Of-Publication"] == 0) |
    (books["Year-Of-Publication"] > current_year),
    "Year-Of-Publication"
] = np.nan

books["ISBN"] = books["ISBN"].astype(str).str.strip()

books["Book-Title"] = (
    books["Book-Title"]
    .fillna("Unknown")
    .astype(str)
    .str.strip()
    .str.title()
)

books["Book-Author"] = (
    books["Book-Author"]
    .fillna("Unknown")
    .astype(str)
    .str.strip()
    .str.title()
)

books["Publisher"] = (
    books["Publisher"]
    .fillna("Unknown")
    .astype(str)
    .str.strip()
    .str.title()
)

In [32]:
books = books.drop(
    columns=["Image-URL-S", "Image-URL-M", "Image-URL-L"]
)

In [33]:
ratings["User-ID"] = pd.to_numeric(
    ratings["User-ID"],
    errors="coerce"
)

ratings["Book-Rating"] = pd.to_numeric(
    ratings["Book-Rating"],
    errors="coerce"
)

ratings["ISBN"] = ratings["ISBN"].astype(str).str.strip()

ratings = ratings.dropna(
    subset=["User-ID", "ISBN", "Book-Rating"]
)

ratings["User-ID"] = ratings["User-ID"].astype(int)

ratings = ratings[
    (ratings["Book-Rating"] >= 1) &
    (ratings["Book-Rating"] <= 10)
]

In [34]:
ratings = ratings[
    ratings["User-ID"].isin(users["User-ID"]) &
    ratings["ISBN"].isin(books["ISBN"])
]

In [35]:
min_ratings = 10

while True:
    previous_count = len(ratings)

    user_counts = ratings["User-ID"].value_counts()
    book_counts = ratings["ISBN"].value_counts()

    ratings = ratings[
        ratings["User-ID"].isin(
            user_counts[user_counts >= min_ratings].index
        ) &
        ratings["ISBN"].isin(
            book_counts[book_counts >= min_ratings].index
        )
    ]

    if len(ratings) == previous_count:
        break

users = users[users["User-ID"].isin(ratings["User-ID"])]
books = books[books["ISBN"].isin(ratings["ISBN"])]

In [36]:
users = users.drop(columns=["ingestion_timestamp"])
books = books.drop(columns=["ingestion_timestamp"])
ratings = ratings.drop(columns=["ingestion_timestamp"])

users.to_parquet("silver/users_silver.parquet", index=False)

books.to_parquet("silver/books_silver.parquet", index=False)

ratings.to_parquet("silver/ratings_silver.parquet", index=False)

In [37]:


print("Silver Layer Datasets:")
print("Users:", users.shape)
print("Books:", books.shape)
print("Ratings:", ratings.shape)

Silver Layer Datasets:
Users: (1820, 5)
Books: (2030, 5)
Ratings: (41456, 3)


**Step 4**

In [38]:
users = pd.read_parquet("silver/users_silver.parquet")
books = pd.read_parquet("silver/books_silver.parquet")
ratings = pd.read_parquet("silver/ratings_silver.parquet")

In [39]:
user_book_matrix = ratings.pivot_table(
    index="ISBN",
    columns="User-ID",
    values="Book-Rating",
    fill_value=0
)

In [40]:
from scipy.sparse import csr_matrix, save_npz

book_ids = pd.Index(books["ISBN"].unique())
user_ids = pd.Index(users["User-ID"].unique())

matrix_ratings = ratings.groupby(
    ["ISBN", "User-ID"],
    as_index=False
)["Book-Rating"].mean()

row_indices = book_ids.get_indexer(matrix_ratings["ISBN"])
col_indices = user_ids.get_indexer(matrix_ratings["User-ID"])

user_book_matrix = csr_matrix(
    (
        matrix_ratings["Book-Rating"].values,
        (row_indices, col_indices)
    ),
    shape=(len(book_ids), len(user_ids))
)

save_npz("gold/user_book_matrix.npz", user_book_matrix)

In [41]:
pd.DataFrame({"ISBN": book_ids}).to_parquet(
    "gold/book_ids.parquet",
    index=False
)

pd.DataFrame({"User-ID": user_ids}).to_parquet(
    "gold/user_ids.parquet",
    index=False
)

In [42]:
book_lookup = books[
    ["ISBN", "Book-Title", "Book-Author"]
].copy()

book_lookup.to_parquet(
    "gold/book_lookup.parquet",
    index=False
)

In [43]:
user_aggregates = ratings.groupby("User-ID").agg(
    average_rating=("Book-Rating", "mean"),
    rating_count=("Book-Rating", "count")
).reset_index()

user_aggregates.to_parquet(
    "gold/user_aggregates.parquet",
    index=False
)

In [44]:
book_aggregates = ratings.groupby("ISBN").agg(
    average_rating=("Book-Rating", "mean"),
    rating_count=("Book-Rating", "count")
).reset_index()

book_aggregates.to_parquet(
    "gold/book_aggregates.parquet",
    index=False
)

In [46]:
from scipy.sparse import load_npz

print("STEP 4: GOLD LAYER COMPLETED\n")

print("Gold Layer Files:")
for file in os.listdir("gold"):
    print("-", file)

matrix = load_npz("gold/user_book_matrix.npz")
book_lookup = pd.read_parquet("gold/book_lookup.parquet")
user_aggregates = pd.read_parquet("gold/user_aggregates.parquet")
book_aggregates = pd.read_parquet("gold/book_aggregates.parquet")

print("Sparse Matrix Shape:", matrix.shape)
print("Book Lookup Shape:", book_lookup.shape)
print("User Aggregates Shape:", user_aggregates.shape)
print("Book Aggregates Shape:", book_aggregates.shape)

STEP 4: GOLD LAYER COMPLETED

Gold Layer Files:
- user_aggregates.parquet
- user_book_matrix.npz
- book_lookup.parquet
- user_ids.parquet
- book_aggregates.parquet
- book_ids.parquet
Sparse Matrix Shape: (2030, 1820)
Book Lookup Shape: (2030, 3)
User Aggregates Shape: (1820, 3)
Book Aggregates Shape: (2030, 3)


**Step 5**

In [48]:
from scipy.sparse import load_npz
from sklearn.neighbors import NearestNeighbors

user_book_matrix = load_npz("gold/user_book_matrix.npz")

book_ids = pd.read_parquet("gold/book_ids.parquet")["ISBN"].tolist()
user_ids = pd.read_parquet("gold/user_ids.parquet")["User-ID"].tolist()

book_lookup = pd.read_parquet("gold/book_lookup.parquet")



In [50]:
model = NearestNeighbors(
    metric="cosine",
    algorithm="brute",
    n_neighbors=6
)

model.fit(user_book_matrix)

print("Cosine similarity Done")

Cosine similarity Done


In [51]:
book_index = {isbn: i for i, isbn in enumerate(book_ids)}

def similar_books(isbn, n=5):

    if isbn not in book_index:
        return pd.DataFrame()

    index = book_index[isbn]

    distances, indices = model.kneighbors(
        user_book_matrix[index],
        n_neighbors=min(n + 1, len(book_ids))
    )

    similar_indices = indices[0][1:]
    similarities = 1 - distances[0][1:]

    result = pd.DataFrame({
        "ISBN": [book_ids[i] for i in similar_indices],
        "Similarity": similarities
    })

    result = result.merge(book_lookup, on="ISBN")

    return result.head(n)

In [52]:
def recommend(user_id, n=10):

    if user_id not in user_ids:
        return pd.DataFrame()

    user_index = user_ids.index(user_id)

    user_ratings = user_book_matrix[:, user_index].toarray().ravel()

    liked_books = np.where(user_ratings >= 7)[0]

    if len(liked_books) == 0:
        return pd.DataFrame()

    scores = {}

    for book_index_value in liked_books:

        rating = user_ratings[book_index_value]

        distances, indices = model.kneighbors(
            user_book_matrix[book_index_value],
            n_neighbors=min(21, len(book_ids))
        )

        for i in range(1, len(indices[0])):

            similar_index = indices[0][i]
            similarity = 1 - distances[0][i]

            if user_ratings[similar_index] == 0 and similarity > 0:

                isbn = book_ids[similar_index]

                scores[isbn] = scores.get(isbn, 0) + (
                    similarity * rating
                )

    if not scores:
        return pd.DataFrame()

    recommendations = pd.DataFrame(
        list(scores.items()),
        columns=["ISBN", "Recommendation Score"]
    )

    recommendations = recommendations.sort_values(
        by="Recommendation Score",
        ascending=False
    )

    recommendations = recommendations.merge(
        book_lookup,
        on="ISBN"
    )

    return recommendations.head(n)

**Step 6**

In [53]:
demo_users = []

for user_id in user_ids:
    user_index = user_ids.index(user_id)
    user_ratings = user_book_matrix[:, user_index].toarray().ravel()

    if np.any(user_ratings >= 7) and not recommend(user_id, n=10).empty:
        demo_users.append(user_id)

    if len(demo_users) == 3:
        break

for user_id in demo_users:
    print("\n" + "=" * 60)
    print("USER ID:", user_id)

    user_index = user_ids.index(user_id)
    user_ratings = user_book_matrix[:, user_index].toarray().ravel()

    liked_indices = np.where(user_ratings >= 7)[0]

    highly_rated_books = pd.DataFrame({
        "ISBN": [book_ids[i] for i in liked_indices],
        "User Rating": user_ratings[liked_indices]
    })

    highly_rated_books = highly_rated_books.merge(
        book_lookup,
        on="ISBN"
    )

    print("\nHighly Rated Books:")
    display(highly_rated_books[["Book-Title", "Book-Author", "User Rating"]].head(10))

    print("\nRecommended Books:")
    display(recommend(user_id, n=10))


USER ID: 243

Highly Rated Books:


,Book-Title,Book-Author,User Rating
0,The God Of Small Things,Arundhati Roy,7.0
1,Memoirs Of A Geisha,Arthur Golden,10.0
2,A Painted House,John Grisham,7.0
3,The Bean Trees,Barbara Kingsolver,10.0
4,Me Talk Pretty One Day,David Sedaris,9.0
5,A Map Of The World,Jane Hamilton,7.0
6,The Pilot'S Wife : A Novel,Anita Shreve,9.0
7,"River, Cross My Heart",Breena Clarke,7.0
8,Unnatural Exposure,Patricia Daniels Cornwell,9.0
9,The General'S Daughter,Nelson Demille,9.0



Recommended Books:


,ISBN,Recommendation Score,Book-Title,Book-Author
0,0452282829,5.903687,We Were The Mulvaneys,Joyce Carol Oates
1,0380730138,4.679092,Vinegar Hill (Oprah'S Book Club (Paperback)),A. Manette Ansay
2,0425162443,4.289902,The Rapture Of Canaan,Sheri Reynolds
3,0345439104,3.579007,Drowning Ruth (Oprah'S Book Club),Christina Schwarz
4,0425147622,3.575107,The Body Farm,Patricia Daniels Cornwell
5,0425154092,3.338380,From Potter'S Field,Patricia Daniels Cornwell
6,0743237188,3.305718,Fall On Your Knees (Oprah #45),Ann-Marie Macdonald
7,0060921145,3.230985,Animal Dreams,Barbara Kingsolver
8,0425169863,3.024232,Point Of Origin,Patricia Daniels Cornwell
9,0671038184,2.971436,Jewel,Bret Lott



USER ID: 254

Highly Rated Books:


,Book-Title,Book-Author,User Rating
0,Animal Farm,George Orwell,8.0
1,The Secret Life Of Bees,Sue Monk Kidd,9.0
2,She'S Come Undone (Oprah'S Book Club),Wally Lamb,7.0
3,American Gods,Neil Gaiman,10.0
4,1984,George Orwell,9.0
5,The Hobbit: Or There And Back Again,J.R.R. Tolkien,9.0
6,Harry Potter And The Sorcerer'S Stone (Book 1),J. K. Rowling,9.0
7,The Bonesetter'S Daughter,Amy Tan,9.0
8,Harry Potter And The Chamber Of Secrets (Book 2),J. K. Rowling,9.0
9,Harry Potter And The Prisoner Of Azkaban (Book 3),J. K. Rowling,9.0


... (2 outputs left out)


USER ID: 638

Highly Rated Books:


,Book-Title,Book-Author,User Rating
0,The Catcher In The Rye,J.D. Salinger,10.0
1,Four Blind Mice,James Patterson,10.0
2,The Lovely Bones: A Novel,Alice Sebold,10.0
3,The Da Vinci Code,Dan Brown,10.0
4,The Rescue,Nicholas Sparks,9.0
5,Bridget Jones : The Edge Of Reason,Helen Fielding,7.0
6,The Pilot'S Wife : A Novel Tag: Author Of The ...,Anita Shreve,10.0
7,Skipping Christmas,John Grisham,10.0
8,The Reader,Bernhard Schlink,10.0
9,Message In A Bottle,Nicholas Sparks,10.0



Recommended Books:


,ISBN,Recommendation Score,Book-Title,Book-Author
0,0399148248,12.379018,Midnight Bayou,Nora Roberts
1,0446532452,9.145685,The Wedding,Nicholas Sparks
2,0553802488,7.442929,The Face,Dean Koontz
3,0553802461,7.069145,Dead Aim,Iris Johansen
4,0743203631,6.948253,Gap Creek: The Story Of A Marriage,Robert Morgan
5,0671525751,6.668393,Someone To Watch Over Me : A Novel,Judith Mcnaught
6,0515134368,6.632610,The Penwyth Curse,Catherine Coulter
7,0399148639,6.628163,Mortal Prey,John Sandford
8,0385421982,6.045920,The Pelican Brief,John Grisham
9,0345452577,5.907251,The Conspiracy Club,Jonathan Kellerman
